# Step 5 — Adjuster-Claim Match Scorer
**Roadmap reference:** Weighted scoring function

## Goal
Replace the rigid routing cascade (zip → state → supervisor fallback) with a
scorer that ranks eligible adjusters for any incoming claim.

## Scoring formula
```
Score = w1 × LOB_match
      + w2 × historical_FTR_rate
      + w3 × (1 − utilisation)
      + w4 × experience_alignment
```

| Component | Weight | Source |
|-----------|--------|--------|
| LOB match (binary) | 0.30 | Adjuster profile |
| Historical FTR rate on similar claims | 0.25 | Performance table |
| Inverse utilisation (1 - active/max) | 0.20 | Capacity table |
| Experience alignment to predicted complexity | 0.25 | Profile + Step 4 output |

## Experience alignment
- Predicted Group 0 / A → Adjuster I / II sufficient → score = 1.0
- Predicted Group B → Specialist preferred → score 0.8 if specialist, 0.5 otherwise
- Predicted Group C → Senior or Specialist required → score 1.0 if senior, 0.6 otherwise

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

DATA = Path("data")
claims      = pd.read_csv(DATA / "step2_feature_matrix.csv")
preds       = pd.read_csv(DATA / "step4_predictions.csv")
perf        = pd.read_csv(DATA / "cp_adjuster_performance.csv")
history     = pd.read_csv(DATA / "step1_tagged_assignments.csv")

claims = claims.merge(preds[["claim_id", "predicted_group"]], on="claim_id", how="left")
print(f"Claims with predictions: {len(claims)}")
perf

Claims with predictions: 200


,adjuster_id,adjuster_name,group_name,cp_claims_handled_2024,ftr_count,historical_ftr_rate,avg_cycle_days,avg_reassignments_per_claim,active_open_claims,max_capacity,utilisation_pct,cp_experience_years,large_loss_certified
0,ADJ-004,Marcus Thompson,Columbus Property Team 3,157,111,0.71,65.9,2.51,31,45,0.689,15,N
1,ADJ-008,Robert Harrington,Pittsburgh Property Team 2,260,169,0.65,80.2,2.65,36,50,0.720,16,N
2,ADJ-014,Rachel Coleman,Senior Adjusters Unit,228,194,0.85,67.4,2.55,10,15,0.667,2,N
3,ADJ-009,Kevin Walsh,CAT Response Team - Central,243,151,0.62,45.0,1.09,82,90,0.911,16,Y
4,ADJ-006,David O'Brien,ISS Special Investigations,168,151,0.90,39.6,3.96,15,20,0.750,11,Y
5,ADJ-001,Sarah Mitchell,West Zone Ops Ohio Team 6,186,126,0.68,38.3,2.77,38,50,0.760,14,N
6,ADJ-015,Christopher Adams,Subrogation Recovery Unit,195,156,0.80,45.3,3.17,20,30,0.667,12,Y


In [2]:
# ── Scoring function ──────────────────────────────────────────────────────────
WEIGHTS = {"lob_match": 0.30, "ftr_rate": 0.25, "inv_util": 0.20, "exp_align": 0.25}

SENIORITY = {
    "ADJ-004": "specialist",  "ADJ-008": "generalist",
    "ADJ-014": "senior",      "ADJ-009": "specialist",
    "ADJ-006": "senior",      "ADJ-001": "generalist",
    "ADJ-015": "specialist",
}

def experience_alignment(predicted_group, adj_id):
    level = SENIORITY.get(adj_id, "generalist")
    if predicted_group in ("0", "A"):
        return 1.0  # any adjuster suitable
    if predicted_group == "B":
        return 1.0 if level in ("specialist", "senior") else 0.5
    # Group C
    return 1.0 if level == "senior" else (0.7 if level == "specialist" else 0.4)

def score_adjuster(adj_row, predicted_group):
    lob_match   = 1.0   # all these adjusters handle CP
    ftr         = adj_row["historical_ftr_rate"]
    inv_util    = 1.0 - adj_row["utilisation_pct"]
    inv_util    = max(0, min(1, inv_util))   # clip
    exp_align   = experience_alignment(predicted_group, adj_row["adjuster_id"])

    return (WEIGHTS["lob_match"]  * lob_match +
            WEIGHTS["ftr_rate"]   * ftr        +
            WEIGHTS["inv_util"]   * inv_util   +
            WEIGHTS["exp_align"]  * exp_align)

# Score all adjusters for each claim
score_rows = []
for _, claim in claims.iterrows():
    ranked = []
    for _, adj in perf.iterrows():
        s = score_adjuster(adj, claim["predicted_group"])
        ranked.append((adj["adjuster_id"], adj["adjuster_name"], round(s, 4)))
    ranked.sort(key=lambda x: x[2], reverse=True)
    top1_id, top1_name, top1_score = ranked[0]
    score_rows.append({
        "claim_id": claim["claim_id"],
        "predicted_group": claim["predicted_group"],
        "final_group": claim["final_group"],
        "top_scorer_id": top1_id,
        "top_scorer_name": top1_name,
        "top_score": top1_score,
        "all_scores": str(ranked),
    })

df_scores = pd.DataFrame(score_rows)
print(f"Scored {len(df_scores)} claims")
print("\nTop scorer distribution:")
print(df_scores["top_scorer_name"].value_counts().to_string())

Scored 200 claims

Top scorer distribution:
top_scorer_name
Rachel Coleman    200


In [3]:
# ── Compare scorer recommendation vs actual final adjuster ────────────────────
final_adj = (history.groupby("claim_id")
             .apply(lambda g: g.sort_values("assignment_num").iloc[-1]["adjuster_id"])
             .reset_index(name="actual_final_adj_id"))

df_scores = df_scores.merge(final_adj, on="claim_id", how="left")
df_scores["top_match_actual"] = (df_scores["top_scorer_id"] == df_scores["actual_final_adj_id"]).astype(int)

match_rate = df_scores["top_match_actual"].mean()
print(f"Scorer top-1 = actual final owner: {match_rate*100:.1f}%")
print(f"Target: >50%")

# By group
by_group = df_scores.groupby("final_group")["top_match_actual"].mean().round(3)
print("\nMatch rate by group:")
print(by_group.to_string())

Scorer top-1 = actual final owner: 13.5%
Target: >50%

Match rate by group:
final_group
0    0.130
A    0.071
B    0.143
C    0.143


C:\Users\SujaySunilNagvekar\AppData\Local\Temp\ipykernel_86200\289365733.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sort_values("assignment_num").iloc[-1]["adjuster_id"])


In [4]:
fig = px.bar(by_group.reset_index(), x="final_group", y="top_match_actual",
             title="Scorer Top-1 Match Rate vs Actual Final Owner (by Group)",
             labels={"final_group": "Reassignment Group", "top_match_actual": "Match Rate"},
             color="final_group",
             category_orders={"final_group": ["0", "A", "B", "C"]})
fig.add_hline(y=0.50, line_dash="dash", line_color="red",
              annotation_text="Target: 50%")
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()

df_scores.to_csv(DATA / "step5_scorer_results.csv", index=False)
print("Saved → data/step5_scorer_results.csv")

Saved → data/step5_scorer_results.csv
